# Smart Bin — YOLOv8 retraining (Google Colab)

**Dataset:** `realwaste1.zip` in Google Drive → inside it a folder `archive` → inside that 4 class folders (`paper`, `plastic`, `food`, `others`).

**Before running:** in Colab, go to **Runtime → Change runtime type → T4 GPU** (training on CPU will be very slow). Then run the cells top to bottom with **Runtime → Run all**.

**What this notebook does:** mounts Drive → finds and unzips the zip → counts images per class and checks them against the expected counts → makes a stratified 70/15/15 train/val/test split → trains `yolov8n-cls` → reports top-1/top-5 accuracy, per-class precision/recall/F1, confusion matrix, a confidence-threshold sweep for the <60% Others/reject route, and single-image latency → copies every output back to Drive.

> Classification, not detection: the dataset is folder-per-class images with no bounding boxes, so the correct metrics are accuracy / precision / recall / F1 and the confusion matrix — **not mAP**.


In [ ]:
import sys, subprocess
print("Python:", sys.version)
# Plain-Python pip install (no !pip magic, so every cell compiles as Python)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "scikit-learn", "matplotlib", "seaborn", "pandas"])
import torch, ultralytics, sklearn
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU detected — set Runtime > Change runtime type > T4 GPU, then re-run.")
print("ultralytics:", ultralytics.__version__)
print("scikit-learn:", sklearn.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
from pathlib import Path
import os

ZIP_NAME = "realwaste1.zip"
# Search the whole of MyDrive for the zip (in case it is inside a folder)
hits = list(Path("/content/drive/MyDrive").rglob(ZIP_NAME))
if not hits:
    raise FileNotFoundError(f"{ZIP_NAME} not found anywhere under MyDrive. Upload it to Drive and re-run.")
ZIP_PATH = hits[0]
print("Found zip:", ZIP_PATH)
print("Size (MB):", round(ZIP_PATH.stat().st_size / 1e6, 1))

In [ ]:
import zipfile
from pathlib import Path

UNZIP_DIR = Path("/content/realwaste_unzipped")
if not (UNZIP_DIR.exists() and any(UNZIP_DIR.iterdir())):
    print("Unzipping", ZIP_PATH, "— this takes a couple of minutes for ~500 MB ...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(UNZIP_DIR)
    print("Done.")
else:
    print("Already unzipped, skipping.")
print("Top level of unzipped folder:")
for p in sorted(UNZIP_DIR.iterdir()):
    print(" -", p.name, "(dir)" if p.is_dir() else "(file)")
# Find the 'archive' folder wherever it sits inside the zip
archives = [p for p in UNZIP_DIR.rglob("archive") if p.is_dir()]
if not archives:
    raise FileNotFoundError("No folder named 'archive' found inside the zip. Check the zip structure and re-run.")
ARCHIVE_DIR = archives[0]
print("\nArchive folder:", ARCHIVE_DIR)
print("Inside archive:")
for p in sorted(ARCHIVE_DIR.iterdir()):
    print(" -", p.name, "(dir)" if p.is_dir() else "(file)")

In [ ]:
from pathlib import Path
from collections import Counter
import json

# --- Config: expected counts (author-supplied, to be checked against the folders) ---
EXPECTED = {"paper": 1831, "plastic": 1674, "food": 1461, "others": 1885}
ORIGINAL_REMAP = {"paper": 500, "plastic": 921, "food": 411, "others": 2920}  # RealWaste 9->4 remap, cardboard -> Others
PHONE_PER_CLASS = 71  # Food confirmed; Paper/Plastic/Others = author recollection, same as Food
IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

# Map each subfolder of archive/ to a class (case-insensitive; also handles an images/ subfolder)
CLASS_NAMES = ["paper", "plastic", "food", "others"]
counts = Counter()
DATA_DIR = {}
for sub in sorted(ARCHIVE_DIR.iterdir()):
    if not sub.is_dir():
        continue
    key = sub.name.strip().lower()
    if key not in CLASS_NAMES:
        print("WARNING: unexpected folder inside archive/ — ignored:", sub.name)
        continue
    img_dir = sub / "images" if (sub / "images").is_dir() else sub
    files = [p for p in img_dir.rglob("*") if p.is_file() and p.suffix.lower() in IMG_EXTS]
    counts[key] = len(files)
    DATA_DIR[key] = img_dir
    print(f"{key:8s} {len(files):5d} images   (from {img_dir})")

print("\nClass        actual   expected   expected-(original+71 phone)")
for c in CLASS_NAMES:
    base = ORIGINAL_REMAP[c] + PHONE_PER_CLASS
    print(f"{c:8s} {counts[c]:7d} {EXPECTED[c]:9d} {EXPECTED[c] - base:+9d}   (original {ORIGINAL_REMAP[c]} + 71 phone = {base})")
total = sum(counts.values())
print(f"\nTOTAL actual: {total}   expected: {sum(EXPECTED.values())}   difference: {total - sum(EXPECTED.values()):+d}")
print(f"RealWaste original total: 4752 | remap + 284 phone photos = 5036 | stated total 6851 leaves +1815 unexplained")
print("If actual != expected, the folder counts win — send the output of this cell back, and the paper numbers get corrected to match.")
Path("/content/smartbin_runs").mkdir(parents=True, exist_ok=True)
with open("/content/smartbin_runs/counts.json", "w") as f:
    json.dump({"actual": dict(counts), "expected": EXPECTED, "original_remap": ORIGINAL_REMAP, "phone_per_class": PHONE_PER_CLASS, "total_actual": total}, f, indent=2)
print("Saved counts to /content/smartbin_runs/counts.json")

In [ ]:
import random, shutil
from pathlib import Path

# --- Stratified 70/15/15 split (seed 42) into train/val/test folders ---
SEED = 42
SPLIT_DIR = Path("/content/smartbin_split")
rng = random.Random(SEED)
if SPLIT_DIR.exists():
    shutil.rmtree(SPLIT_DIR)
for split in ["train", "val", "test"]:
    for c in CLASS_NAMES:
        (SPLIT_DIR / split / c).mkdir(parents=True, exist_ok=True)
split_counts = {}
for c in CLASS_NAMES:
    files = sorted(DATA_DIR[c].rglob("*"))
    files = [p for p in files if p.is_file() and p.suffix.lower() in IMG_EXTS]
    rng.shuffle(files)
    n = len(files)
    n_train, n_val = int(0.70 * n), int(0.15 * n)
    parts = {"train": files[:n_train], "val": files[n_train:n_train + n_val], "test": files[n_train + n_val:]}
    split_counts[c] = {}
    for split, fs in parts.items():
        for src in fs:
            shutil.copy2(src, SPLIT_DIR / split / c / src.name)
        split_counts[c][split] = len(fs)
print("Split (train/val/test):")
for c in CLASS_NAMES:
    print(f"  {c:8s}", split_counts[c])
print("\nSplit folder:", SPLIT_DIR)

In [ ]:
from ultralytics import YOLO
import torch

# --- Training ---
EPOCHS = 30
IMGSZ = 224
BATCH = 32
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Training on:", "GPU" if DEVICE == 0 else "CPU (slow — switch to a T4 GPU runtime)")
model = YOLO("yolov8n-cls.pt")  # downloads the pretrained weights on first run
model.train(data=str(SPLIT_DIR), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                      device=DEVICE, project="/content/smartbin_runs", name="yolov8n_cls_4class",
                      seed=SEED, verbose=False)
print("Training finished. Best weights: /content/smartbin_runs/yolov8n_cls_4class/weights/best.pt")

In [ ]:
from pathlib import Path
import json
from ultralytics import YOLO

# --- Evaluate on val and test (top-1 / top-5) ---
best = Path("/content/smartbin_runs/yolov8n_cls_4class/weights/best.pt")
model = YOLO(str(best))
metrics = {}
for split in ["val", "test"]:
    m = model.val(data=str(SPLIT_DIR), split=split, verbose=False)
    metrics[split] = {"top1": float(m.top1), "top5": float(m.top5)}
    print(f"{split}: top-1 = {m.top1:.4f}   top-5 = {m.top5:.4f}")
with open("/content/smartbin_runs/paper_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("Saved /content/smartbin_runs/paper_metrics.json")

In [ ]:
from pathlib import Path
import numpy as np
from ultralytics import YOLO
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import json

# --- Per-class report + confusion matrix on the TEST split ---
best = Path("/content/smartbin_runs/yolov8n_cls_4class/weights/best.pt")
model = YOLO(str(best))
test_dir = Path("/content/smartbin_split/test")
y_true, y_pred, y_conf, records = [], [], [], []
for c in CLASS_NAMES:
    for img in sorted(test_dir.glob(f"{c}/*")):
        if img.suffix.lower() not in IMG_EXTS:
            continue
        r = model(str(img), verbose=False)[0]
        pred = r.names[r.probs.top1]
        conf = float(r.probs.top1conf.item())
        y_true.append(c); y_pred.append(pred); y_conf.append(conf)
        records.append({"file": img.name, "true": c, "pred": pred, "conf": conf})
print(classification_report(y_true, y_pred, labels=CLASS_NAMES, digits=4))
cm = confusion_matrix(y_true, y_pred, labels=CLASS_NAMES)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, cmap="Blues")
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Confusion matrix — test split (YOLOv8n-cls)")
plt.tight_layout()
plt.savefig("/content/smartbin_runs/confusion_matrix.png", dpi=150)
plt.show()
with open("/content/smartbin_runs/test_predictions.json", "w") as f:
    json.dump(records, f, indent=2)
print(f"Saved confusion_matrix.png and test_predictions.json ({len(records)} test images)")

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path
import random

# --- Sample predictions (12 random test images with true vs predicted label) ---
best = Path("/content/smartbin_runs/yolov8n_cls_4class/weights/best.pt")
model = YOLO(str(best))
test_dir = Path("/content/smartbin_split/test")
all_imgs = [(c, p) for c in CLASS_NAMES for p in sorted(test_dir.glob(f"{c}/*")) if p.suffix.lower() in IMG_EXTS]
sample = random.Random(7).sample(all_imgs, min(12, len(all_imgs)))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for ax, (true_c, img_path) in zip(axes.flat, sample):
    r = model(str(img_path), verbose=False)[0]
    pred = r.names[r.probs.top1]; conf = float(r.probs.top1conf.item()) * 100
    ax.imshow(plt.imread(img_path)); ax.axis("off")
    ax.set_title(f"true: {true_c}\npred: {pred} ({conf:.0f}%)", fontsize=9,
                 color="green" if pred == true_c else "red")
plt.tight_layout()
plt.savefig("/content/smartbin_runs/sample_predictions.png", dpi=150)
plt.show()
print("Saved sample_predictions.png")

In [ ]:
import numpy as np
import json

# --- Confidence-threshold sweep: below threshold -> Others (reject route) ---
# The FYP report used 0.60 as the reject threshold; this sweep checks whether it is the best one.
thresholds = [0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90]
rows = []
for t in thresholds:
    routed = [("others" if conf < t else pred) for pred, conf in zip(y_pred, y_conf)]
    acc = float(np.mean([a == b for a, b in zip(y_true, routed)]))
    reject = float(np.mean([conf < t for conf in y_conf]))
    rows.append({"threshold": t, "accuracy_with_reject": round(acc, 4), "reject_rate": round(reject, 4)})
    print(f"threshold {t:.2f}: accuracy {acc:.4f}   reject rate {reject:.3f}")
with open("/content/smartbin_runs/threshold_sweep.json", "w") as f:
    json.dump(rows, f, indent=2)
print("Saved threshold_sweep.json — report the 0.60 row in the paper, and note the best threshold here.")

In [ ]:
import time
import numpy as np
from pathlib import Path

# --- Single-image latency (mean / p95) on 20 test images ---
test_dir = Path("/content/smartbin_split/test")
imgs = [p for c in CLASS_NAMES for p in sorted(test_dir.glob(f"{c}/*")) if p.suffix.lower() in IMG_EXTS][:20]
times = []
for img in imgs:
    t0 = time.perf_counter()
    model(str(img), verbose=False)
    times.append((time.perf_counter() - t0) * 1000)
print(f"Images timed: {len(times)} | device: {'GPU' if DEVICE == 0 else 'CPU'}")
print(f"Mean latency: {np.mean(times):.0f} ms | p95: {np.percentile(times, 95):.0f} ms | min: {np.min(times):.0f} ms | max: {np.max(times):.0f} ms")
print("PC/GPU model (for the paper):", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU-only Colab runtime")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# --- Graph 1: Training curves — loss + val top-1 accuracy vs epoch (from results.csv) ---
csv_path = Path("/content/smartbin_runs/yolov8n_cls_4class/results.csv")
df = pd.read_csv(csv_path)
df.columns = [c.strip() for c in df.columns]
print("Columns in results.csv:", df.columns.tolist())
loss_cols = [c for c in df.columns if "loss" in c and "train" in c]
acc_cols = [c for c in df.columns if "accuracy_top1" in c]
if not loss_cols or not acc_cols:
    raise ValueError("Loss/accuracy column not found. Columns are: " + str(df.columns.tolist()))
loss_col, acc_col = loss_cols[0], acc_cols[0]
print("Using columns:", loss_col, "|", acc_col)
print(df[["epoch", loss_col, acc_col]].tail())
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(df["epoch"], df[loss_col], marker="o", label="Train loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Classification loss")
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(df["epoch"], df[acc_col] * 100, marker="s", color="green", label="Val top-1 acc (%)")
ax2.set_ylabel("Val top-1 accuracy (%)")
ax1.set_title("Training curves — YOLOv8n-cls (30 epochs)")
fig.tight_layout()
plt.savefig("/content/smartbin_runs/training_curves.png", dpi=150)
plt.show()
print("Saved training_curves.png")

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report

# --- Graph 2: Per-class precision / recall / F1 bar chart (from test predictions) ---
CLASS_NAMES = ["paper", "plastic", "food", "others"]
recs = json.load(open("/content/smartbin_runs/test_predictions.json"))
y_true = [r["true"] for r in recs]
y_pred = [r["pred"] for r in recs]
print("Test predictions loaded:", len(recs))
rep = classification_report(y_true, y_pred, labels=CLASS_NAMES, output_dict=True, digits=4)
prec = [rep[c]["precision"] for c in CLASS_NAMES]
rec = [rep[c]["recall"] for c in CLASS_NAMES]
f1 = [rep[c]["f1-score"] for c in CLASS_NAMES]
print(f"{'class':8s} {'P':>6s} {'R':>6s} {'F1':>6s}")
for c, a, b, d in zip(CLASS_NAMES, prec, rec, f1):
    print(f"{c:8s} {a:6.4f} {b:6.4f} {d:6.4f}")
x = np.arange(len(CLASS_NAMES)); w = 0.25
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w, prec, w, label="Precision")
ax.bar(x, rec, w, label="Recall")
ax.bar(x + w, f1, w, label="F1-score")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylim(0.75, 1.0); ax.set_ylabel("Score"); ax.set_title("Per-class metrics — test split")
ax.legend(); ax.grid(True, axis="y", alpha=0.3)
for bars in ax.containers:
    ax.bar_label(bars, fmt="%.3f", fontsize=8)
plt.tight_layout()
plt.savefig("/content/smartbin_runs/per_class_metrics.png", dpi=150)
plt.show()
print("Saved per_class_metrics.png")

In [ ]:
import json
import matplotlib.pyplot as plt

# --- Graph 3: Threshold sweep — accuracy vs reject rate (0.60 = the FYP reject rule) ---
rows = json.load(open("/content/smartbin_runs/threshold_sweep.json"))
print("Sweep rows:", rows)
ths = [r["threshold"] for r in rows]
acc = [r["accuracy_with_reject"] * 100 for r in rows]
rej = [r["reject_rate"] * 100 for r in rows]
i60 = min(range(len(ths)), key=lambda i: abs(ths[i] - 0.60))
fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(ths, acc, marker="o", label="Accuracy with reject (%)")
ax1.axvline(ths[i60], color="red", linestyle="--", label="0.60 (used in system)")
ax1.set_xlabel("Confidence threshold")
ax1.set_ylabel("Accuracy (%)")
ax1.grid(True, alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(ths, rej, marker="s", color="orange", label="Reject rate (%)")
ax2.set_ylabel("Reject rate (%)")
ax1.set_title("Confidence-threshold sweep — test split")
ax1.legend(loc="lower left")
fig.tight_layout()
plt.savefig("/content/smartbin_runs/threshold_sweep.png", dpi=150)
plt.show()
print("Saved threshold_sweep.png — at 0.60: accuracy", acc[i60], "%, reject rate", rej[i60], "%")

In [ ]:
import shutil, datetime
from pathlib import Path

# --- Copy every output back to Drive ---
stamp = datetime.datetime.now().strftime("%Y-%m-%d")
OUT = Path(f"/content/drive/MyDrive/SmartBin_Results_{stamp}")
OUT.mkdir(parents=True, exist_ok=True)
for src in [Path("/content/smartbin_runs/counts.json"),
            Path("/content/smartbin_runs/paper_metrics.json"),
            Path("/content/smartbin_runs/threshold_sweep.json"),
            Path("/content/smartbin_runs/test_predictions.json"),
            Path("/content/smartbin_runs/confusion_matrix.png"),
            Path("/content/smartbin_runs/sample_predictions.png"),
            Path("/content/smartbin_runs/training_curves.png"),
            Path("/content/smartbin_runs/per_class_metrics.png"),
            Path("/content/smartbin_runs/threshold_sweep.png")]:
    if src.exists():
        shutil.copy2(src, OUT / src.name)
        print("copied:", src.name)
run_dir = Path("/content/smartbin_runs/yolov8n_cls_4class")
if run_dir.exists():
    shutil.copytree(run_dir, OUT / "yolov8n_cls_4class", dirs_exist_ok=True)
    print("copied: training run folder (incl. best.pt and results.csv)")
print("\nAll outputs are in Drive at:", OUT)
print("Also keep the notebook itself: File > Save a copy in Drive.")

## What to send back (for the paper)

From the Drive folder `SmartBin_Results_<date>` (or from `/content/smartbin_runs` in the session), send:

1. **`counts.json`** — the output of the count cell (this settles the 6,851 vs 4,752 question).
2. **`paper_metrics.json`** — top-1 / top-5 on val and test.
3. **`confusion_matrix.png`** and **`sample_predictions.png`**.
3b. **`training_curves.png`**, **`per_class_metrics.png`** and **`threshold_sweep.png`** (the 3 new graph cells).
4. **`threshold_sweep.json`** — especially the 0.60 row.
5. **`yolov8n_cls_4class/results.csv`** — the training curves.
6. The **latency output** (mean / p95) and whether the run was on a T4 GPU or CPU.

> If the count cell finds a different folder layout (e.g. the 4 folders are named differently, or each has an `images/` subfolder), it prints a warning — send that output as it is, and the notebook will be adjusted to match.
